# Production RAG: Qdrant + BAML + Langfuse

This notebook implements a fully structured, locally-hosted, and observable RAG pipeline.
- **Vector Store:** Qdrant (Dense + Sparse Fusion).
- **Prompting & Output Schema:** BAML (Type-safe structured JSON generation).
- **Observability:** Langfuse (Tracing retrievals and LLM generation).
- **Generation:** Gemma 4 via LM Studio.

In [ ]:
!pip install -qU qdrant-client fastembed sentence-transformers langchain-openai baml-py langfuse

### 1. Configure Langfuse Observability

In [ ]:
import os

# Set your Langfuse credentials to enable telemetry tracing.
# If you do not have an account, you can create a free one at cloud.langfuse.com
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."  # Replace with your public key
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."  # Replace with your secret key
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com" # Or your self-hosted URL

from langfuse.decorators import observe

### 2. Define BAML Schemas and Prompts

We create a directory for our BAML configuration and write out the `.baml` file. This replaces LangChain's PromptTemplates. It explicitly points to our local LM Studio endpoint and defines the structured Pydantic object we want back.

In [ ]:
!mkdir -p baml_src

In [ ]:
%%writefile baml_src/main.baml
generator target {
  type "python"
  output_dir "../baml_client"
}

client<llm> LocalGemma {
  provider openai
  options {
    base_url "http://localhost:1234/v1"
    api_key "lm-studio"
    model "gemma-4"
  }
}

class RAGAnswer {
  is_answerable bool @description("True if the provided context explicitly contains the answer.")
  answer string @description("The final answer. If is_answerable is false, explain why.")
  confidence float @description("A confidence score between 0.0 and 1.0")
}

function GenerateAnswer(context: string, question: string) -> RAGAnswer {
  client LocalGemma
  prompt #"
    You are an expert analytical assistant. 
    Answer the user's question based strictly on the following context.
    
    Context:
    {{ context }}

    Question:
    {{ question }}

    {{ ctx.output_format }}
  "#
}

In [ ]:
# Compile the BAML file into native Python code (this creates the 'baml_client' folder)
!baml-cli generate

### 3. Initialize Local Embedding and Reranking Models

In [ ]:
from fastembed import SparseTextEmbedding
from sentence_transformers import CrossEncoder
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient, models

sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

dense_embeddings = OpenAIEmbeddings(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="text-embedding-nomic" 
)

### 4. Setup Qdrant and Ingest Data

In [ ]:
client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "hybrid_documents"

# Recreate the collection
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": models.VectorParams(size=768, distance=models.Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

# Ingest Documents
documents = [
    "The Apollo 11 mission landed humans on the Moon in 1969. Neil Armstrong was the first to step on the surface.",
    "Project Artemis is NASA's current program to return humans to the Moon and eventually send them to Mars.",
    "The Saturn V was the super heavy-lift launch vehicle used by NASA between 1967 and 1973.",
    "SpaceX's Starship is a fully reusable spacecraft designed to carry both crew and cargo to Earth orbit, the Moon, and Mars.",
    "Hybrid search combines semantic similarity with keyword matching to retrieve the most relevant documents."
]

dense_vectors = dense_embeddings.embed_documents(documents)
sparse_vectors = list(sparse_model.embed(documents))

points = []
for i, doc in enumerate(documents):
    points.append(
        models.PointStruct(
            id=i + 1,
            vector={
                "dense": dense_vectors[i],
                "sparse": models.SparseVector(indices=sparse_vectors[i].indices.tolist(), values=sparse_vectors[i].values.tolist())
            },
            payload={"chunk_text": doc}
        )
    )
client.upsert(collection_name=COLLECTION_NAME, points=points)
print("Ingestion complete.")

### 5. Orchestrate RAG with Langfuse Observability

In [ ]:
# Import the generated BAML client and types
from baml_client.sync_client import b
from baml_client.types import RAGAnswer

# The @observe decorator wraps this function to automatically send telemetry to Langfuse
@observe(as_type="retrieval")
def retrieve_and_rerank(query: str, top_k_initial: int = 10, top_k_final: int = 3):
    dense_query = dense_embeddings.embed_query(query)
    sparse_query_obj = list(sparse_model.embed([query]))[0]
    sparse_query = models.SparseVector(
        indices=sparse_query_obj.indices.tolist(),
        values=sparse_query_obj.values.tolist()
    )

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(query=dense_query, using="dense", limit=top_k_initial),
            models.Prefetch(query=sparse_query, using="sparse", limit=top_k_initial),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k_initial,
        with_payload=True
    )
    
    initial_docs = [hit.payload["chunk_text"] for hit in results.points]
    if not initial_docs:
        return []

    rerank_pairs = [[query, doc] for doc in initial_docs]
    ce_scores = cross_encoder.predict(rerank_pairs)

    scored_docs = list(zip(initial_docs, ce_scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    return [doc for doc, _ in scored_docs[:top_k_final]]

# Observe the BAML execution layer
@observe(as_type="generation")
def generate_baml_answer(query: str, context: str) -> RAGAnswer:
    # BAML handles the prompt templating, LLM execution, and Pydantic validation
    return b.GenerateAnswer(context=context, question=query)

# The parent observation trace
@observe()
def execute_rag(question: str):
    print(f"Executing RAG for: '{question}'")
    
    best_chunks = retrieve_and_rerank(question)
    if not best_chunks:
         print("No relevant context found.")
         return None
         
    context_string = "\n\n---\n\n".join(best_chunks)
    
    # Get the type-safe Pydantic object back from BAML
    response_obj = generate_baml_answer(query=question, context=context_string)
    
    return response_obj

In [ ]:
# Test the pipeline
test_query = "Which vehicle took astronauts to the Moon, and what are its dates of operation?"
final_result = execute_rag(test_query)

print("\n--- Final Structured Output ---")
print(f"Answerable: {final_result.is_answerable}")
print(f"Answer:     {final_result.answer}")
print(f"Confidence: {final_result.confidence}")

# Log into your Langfuse dashboard to see the full trace!